In [44]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, inspect, text
import re
import urllib
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

In [47]:
engine = create_engine(
    r'mssql+pyodbc://DESKTOP-LEVIKVU\SQLEXPRESS/Airbnb?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes', 
    fast_executemany=True
)

In [14]:
raw_data_folder_listing = r"C:\Users\DELL\Desktop\Programming\Data analytics\work\Airbnb\DataAirbnb\listings"
for filename in os.listdir(raw_data_folder_listing):
    if filename.endswith(".csv"):
        file_path = os.path.join(raw_data_folder_listing, filename)
        c = filename.find("Listing")
        city = filename[:c]
        df = pd.read_csv(file_path, low_memory=False)
        df = df.drop(['scrape_id','last_scraped','picture_url','host_thumbnail_url','host_picture_url','calendar_last_scraped',],axis=1)
        df['city'] = city
        inspector = inspect(engine)
        
        if inspector.has_table("listing"):
            
            with engine.connect() as connection:
                query = text(f"SELECT TOP 1 city FROM listing WHERE city = '{city}'")
                result = connection.execute(query).fetchone()
                
                if result is not None:
                    print(f"[{city}] already exists in the database! Skipping file to prevent duplicates.\n")
                    continue # <--- The 'continue' command tells Python to immediately jump to the next file in the folder!
        

        print(f"[{city}] New city found. Reading CSV...")
        df = pd.read_csv(file_path, low_memory=False)
        
        df = df.drop(['scrape_id','last_scraped','source','picture_url','host_thumbnail_url','host_picture_url','calendar_last_scraped'], axis=1, errors='ignore')
        
        df['city'] = city

[Amsterdam] already exists in the database! Skipping file to prevent duplicates.

[HongKong] already exists in the database! Skipping file to prevent duplicates.

[Rome] already exists in the database! Skipping file to prevent duplicates.



In [46]:
#df['price']


clean_string = re.sub(r'\$', '', df['price'][0])

final_int = int(float(clean_string))


def clean_price(price):
    clean_string = re.sub(r'\$', '', df['price'])
    return int(float(clean_string))

clean_price(df['price'])

In [169]:

raw_data_folder_listing = r"C:\Users\DELL\Desktop\Programming\Data analytics\work\Airbnb\DataAirbnb\copy_data\listings"
for filename in os.listdir(raw_data_folder):
    if filename.endswith(".csv"):
        file_path = os.path.join(raw_data_folder_listing, filename)
        c = filename.find("Listing")
        city = filename[:c]
        df = pd.read_csv(file_path, low_memory=False)
        df = df.drop(['scrape_id','last_scraped','picture_url','host_thumbnail_url','host_picture_url','calendar_last_scraped',],axis=1)
        df['city'] = city
        inspector = inspect(engine)
        
        if inspector.has_table("listing"):
            
            with engine.connect() as connection:
                query = text(f"SELECT TOP 1 city FROM listing WHERE city = '{city}'")
                result = connection.execute(query).fetchone()
                
                if result is not None:
                    print(f"[{city}] already exists in the database! Skipping file to prevent duplicates.\n")
                    continue # <--- The 'continue' command tells Python to immediately jump to the next file in the folder!
        

        print(f"[{city}] New city found. Reading CSV...")
        df = pd.read_csv(file_path, low_memory=False)
        
        df = df.drop(['scrape_id','last_scraped','source','picture_url','host_thumbnail_url','host_picture_url','calendar_last_scraped'], axis=1, errors='ignore')
        
        df['city'] = city
        
        print(f'File {filename} is being pushed into SQL database....')
        df.to_sql("listing", con=engine, if_exists='append', index=False, chunksize=100000)
        print(f'File {filename} successfully loaded.\n')

print("Pipeline Complete!")
        

[Amsterdam] New city found. Reading CSV...
File AmsterdamListing.csv is being pushed into SQL database....
File AmsterdamListing.csv successfully loaded.

[HongKong] New city found. Reading CSV...
File HongKongListings.csv is being pushed into SQL database....
File HongKongListings.csv successfully loaded.

[Rome] New city found. Reading CSV...
File RomeListings.csv is being pushed into SQL database....
File RomeListings.csv successfully loaded.

Pipeline Complete!


In [40]:

raw_data_folder_calendar = r"C:\Users\DELL\Desktop\Programming\Data analytics\work\Airbnb\DataAirbnb\copy_data\calendar"

print("Reviews Pipeline Started. Scanning for files...\n")
inspector = inspect(engine)

for filename in os.listdir(raw_data_folder_calendar):
    if filename.endswith(".csv") and "Calendar" in filename:
        file_path = os.path.join(raw_data_folder_calendar, filename)
        
        c = filename.find("Calendar")
        city = filename[:c]

        if inspector.has_table("calendar"):
            with engine.connect() as connection:
                query = text(f"SELECT TOP 1 city FROM calendar WHERE city = '{city}'")
                result = connection.execute(query).fetchone()
                
                if result is not None:
                    print(f" [{city}] already exists in the Calendar table! Skipping.\n")
                    continue

        print(f"[{city}] New city found. Reading CSV...")
        df = pd.read_csv(file_path, low_memory=False)

        df['date'] = pd.to_datetime(df['date'])
        
        df['city'] = city
        
        #df['price'] = df['price'].replace({r'\$': '', ',': ''}, regex=True).astype(float)
        #df['adjusted_price'] = df['adjusted_price'].replace({r'\$': '', ',': ''}, regex=True).astype(float)
        
        print(f'File {filename} is being pushed into SQL database....')
        df.to_sql("calendar", con=engine, if_exists='append', index=False, chunksize=100000)
        print(f'File {filename} successfully loaded.\n')

print("Pipeline Complete!")

Loading AI Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Reviews Pipeline Started. Scanning for files...



OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [53].  (53) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (53)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [17]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [49]:
raw_data_folder_reviews = r"C:\Users\DELL\Desktop\Programming\Data analytics\work\Airbnb\DataAirbnb\reviews"

print("Loading AI Model...")
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("\nReviews Pipeline Started. Scanning for files...\n")
inspector = inspect(engine)

for filename in os.listdir(raw_data_folder_reviews):
    if filename.endswith(".csv") and "Reviews" in filename:
        file_path = os.path.join(raw_data_folder_reviews, filename)
        
        c = filename.find("Reviews")
        city = filename[:c]

        # --- DATABASE CHECK (Matches your Calendar logic exactly) ---
        if inspector.has_table("reviews"):
            with engine.connect() as connection:
                query = text(f"SELECT TOP 1 city FROM reviews WHERE city = '{city}'")
                result = connection.execute(query).fetchone()
                
                if result is not None:
                    print(f"[{city}] already exists in the Reviews table! Skipping.\n")
                    continue

        print(f"[{city}] New city found. Reading CSV...")
        df = pd.read_csv(file_path, low_memory=False)

        df['date'] = pd.to_datetime(df['date'])
        df['city'] = city
        df['unique_listing_id'] = city + "_" + df['listing_id'].astype(str)

        df = df.sort_values('date', ascending=False)
        df_sample = df.head(10000).copy()
        
        res = {}
        print(f"[{city}] Running RoBERTa on the {len(df_sample)} most recent reviews. Grab a coffee...")
        
        for i, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
            text_str = str(row['comments']) 
            myid = row['id'] 
            
            try:
                encoded_text = tokenizer(text_str, return_tensors='pt', truncation=True, max_length=512)
                output = model(**encoded_text)
                
                scores = output[0][0].detach().numpy()
                scores = softmax(scores)
                
                res[myid] = {
                    'roberta_neg': scores[0],
                    'roberta_neu': scores[1],
                    'roberta_pos': scores[2]
                }
            except Exception as e:
                pass
        
        roberta_df = pd.DataFrame(res).T
        roberta_df = roberta_df.reset_index().rename(columns={'index': 'id'})
        
        final_df = df_sample.merge(roberta_df, how='left', on='id')
        
        print(f'File {filename} is being pushed into SQL database....')
        
        final_df.to_sql("reviews", con=engine, if_exists='append', index=False, chunksize=100000)
        
        print(f'File {filename} successfully loaded.\n')

print("Pipeline Complete!")

Loading AI Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Reviews Pipeline Started. Scanning for files...

[Amsterdam] New city found. Reading CSV...
[Amsterdam] Running RoBERTa on the 10000 most recent reviews. Grab a coffee...


  0%|          | 0/10000 [00:00<?, ?it/s]

File AmsterdamReviews.csv is being pushed into SQL database....
File AmsterdamReviews.csv successfully loaded.

[HongKong] New city found. Reading CSV...
[HongKong] Running RoBERTa on the 10000 most recent reviews. Grab a coffee...


  0%|          | 0/10000 [00:00<?, ?it/s]

File HongKongReviews.csv is being pushed into SQL database....
File HongKongReviews.csv successfully loaded.

[Rome] New city found. Reading CSV...
[Rome] Running RoBERTa on the 10000 most recent reviews. Grab a coffee...


  0%|          | 0/10000 [00:00<?, ?it/s]

File RomeReviews.csv is being pushed into SQL database....
File RomeReviews.csv successfully loaded.

Pipeline Complete!
